In [1]:
from pathlib import Path

from neo4j import GraphDatabase, RoutingControl


URI = "neo4j://localhost:7687"
db_name = "rdftest"
AUTH = ("neo4j", "testingpass")
test_nt= Path("../benchmarks/data/bsbm_1/dataset_encoded.nt")
test_nt.exists()

True

In [9]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # driver.execute_query("CALL n10s.graphconfig.init();")
    driver.execute_query("CREATE CONSTRAINT n10s_unique_uri IF NOT EXISTS  FOR (r:Resource) REQUIRE r.uri IS UNIQUE;")


In [10]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    # delete existing data
    driver.execute_query("MATCH (n) DETACH DELETE n;")
    result = driver.execute_query(f'CALL n10s.rdf.import.fetch("file://{test_nt.absolute()}", "Turtle");')
result.summary.metadata


{'query': 'CALL n10s.rdf.import.fetch("file:///nfsd/gracedata2/kantz/Dense-Vector-KG/benchmarks/../benchmarks/data/bsbm_1/dataset_encoded.nt", "Turtle");',
 'parameters': {},
 'server': <neo4j.api.ServerInfo at 0x7f2eb1fdbb90>,
 'database': None,
 't_first': 67,
 'fields': ['terminationStatus',
  'triplesLoaded',
  'triplesParsed',
  'namespaces',
  'extraInfo',
  'callParams'],
 'qid': 0,
 'statuses': [{'gql_status': '00000',
   'status_description': 'note: successful completion',
   'diagnostic_record': {'OPERATION': '',
    'OPERATION_CODE': '0',
    'CURRENT_SCHEMA': '/'}}],
 'type': 'rw',
 't_last': 948,
 'db': 'test'}

In [11]:
from utils.datasets.data_tensor import DataTensor

In [12]:
# iterate over all *_embedding properties and replace them by *_embedding_vector properties containing the same values as lists instead of strings
import tqdm


with GraphDatabase.driver(URI, auth=AUTH) as driver:
    result = driver.execute_query("MATCH (n) RETURN DISTINCT keys(n) AS keys;")
    all_keys: set[str] = set()
    for record in result.records:
        all_keys.update(record.data()["keys"])
    embedding_keys = [key for key in all_keys if key.endswith("_embedding")]
    print(f"Found {len(embedding_keys)} embedding keys: {embedding_keys}")
    g = tqdm.tqdm(desc="Processing records", unit="record")
    for key in embedding_keys:
        vector_key = key.replace("_embedding", "_embedding_vector")
        for record in driver.execute_query(
            f"MATCH (n) WHERE n.{key} IS NOT NULL RETURN n.{key} AS value, elementId(n) AS id"
        ).records:
            value = record["value"]
            if not isinstance(value, str):
                print(f"Warning: value of {key} is not a string: {value}")
                continue
            try:
                vector = DataTensor.from_literal(value).data
                driver.execute_query(
                    f"MATCH (n) WHERE elementId(n) = '{record['id']}' SET n.{vector_key} = {vector};"
                )
                g.update(1)
            except Exception as e:
                print(f"Error parsing value of {key}: {e}")
                continue

Found 2 embedding keys: ['rdfs__comment_embedding', 'rdfs__label_embedding']


Processing records: 603record [00:13, 56.24record/s]

In [28]:
# test a knn query


with GraphDatabase.driver(URI, auth=AUTH) as driver:
    knn = driver.execute_query("""
MATCH (n)-[r:ns1__productFeature]->()
WHERE n.rdfs__comment_embedding_vector IS NOT NULL
WITH n, vector.similarity.euclidean($query, n.rdfs__comment_embedding_vector) AS score
RETURN n, score
ORDER BY score DESCENDING
LIMIT 5
""", query=vector)
    for record in knn.records:
        print(record.data())

{'n': {'rdfs__label_embedding': '{"data": [-0.029847441241145134, 0.040876299142837524, 0.008679131977260113, -0.010968274436891079, -0.029981492087244987, -0.02483729086816311, -0.046693459153175354, 0.03719821572303772, -0.06204835698008537, -0.024038800969719887, 0.037824053317308426, -0.004494285210967064, 0.002616061130538583, -0.015280290506780148, -0.03750353306531906, -0.031264789402484894, 0.06308870762586594, 0.038045480847358704, -0.041160453110933304, 0.020179508253932, -0.035287268459796906, 0.026539256796240807, -0.06246509402990341, 0.10865858942270279, 0.06646935641765594, 0.027944359928369522, -0.03438600152730942, 0.015588706359267235, 0.03937765210866928, -0.07723905146121979, 0.022507773712277412, 0.02336030825972557, 0.01606333814561367, -0.06864223629236221, 0.08392147719860077, 0.04686134308576584, 0.0272365715354681, -0.013164958916604519, 0.04734942689538002, 0.006273831240832806, 0.01575450226664543, 0.016154104843735695, -0.1319909393787384, 0.041752483695745

In [26]:
record.data()['n']

{'rdfs__label_embedding': '{"data": [-0.1617465317249298, -0.08583834022283554, 0.0077483318746089935, -0.019290046766400337, -0.03710947558283806, 0.02820422686636448, 0.023206060752272606, -0.049596142023801804, -0.08355984836816788, -0.016989540308713913, -0.04401499405503273, -0.011290193535387516, -0.06777409464120865, 0.027621455490589142, -0.010528596118092537, 0.016549356281757355, 0.08401678502559662, -0.007730415556579828, 0.01725403033196926, -0.04112978279590607, -0.0864202082157135, -0.025009701028466225, 0.06425581127405167, -0.016849961131811142, -0.04984765499830246, -0.05299564450979233, -0.01660853624343872, 0.022523239254951477, 0.01854400709271431, -0.09636886417865753, -0.037346452474594116, 0.006083080545067787, 0.052814461290836334, 0.004960332065820694, 0.001487561035901308, -0.00036467210156843066, 0.08433394879102707, -0.0698123648762703, 0.08880085498094559, -0.0031968294642865658, 0.036670465022325516, -0.02086496539413929, 0.01604430191218853, -0.0143206520

In [18]:
# create a vector index on the rdfs__comment_embedding_vector property

with GraphDatabase.driver(URI, auth=AUTH) as driver:
    for key in embedding_keys:
        vector_key = key.replace("_embedding", "_embedding_vector")
        driver.execute_query(f"""DROP INDEX {vector_key}_index IF EXISTS;""")
        driver.execute_query(f"""CREATE VECTOR INDEX {vector_key}_index IF NOT EXISTS
                             FOR (n:Resource) ON n.{vector_key} OPTIONS {{
                             indexConfig: {{
                                }}
                             }};""")

2026-05-11 20:33:55,312 - INFO - Received notification from DBMS server: <GqlStatusObject gql_status='00NA1', status_description="note: successful completion - index or constraint does not exist. The command 'DROP INDEX rdfs__label_embedding_vector_index IF EXISTS' has no effect. The specified index or constraint `rdfs__label_embedding_vector_index` does not exist.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'DROP INDEX rdfs__label_embedding_vector_index IF EXISTS;'


In [21]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    for key in embedding_keys:
        print(f"Testing knn query for {key}...")
        vector_key = key.replace("_embedding", "_embedding_vector")
        knn = driver.execute_query(f"""MATCH (n:Resource)
    SEARCH n IN (
    VECTOR INDEX {vector_key}_index
    FOR {vector}
    LIMIT 5
    ) SCORE AS score
    RETURN n, score""")
        for record in knn.records:
            print(record.data())

Testing knn query for rdfs__comment_embedding...
{'n': {'rdfs__label_embedding': '{"data": [-0.058632392436265945, 0.06441012024879456, -0.0029306048527359962, -0.02309849113225937, 0.03482859581708908, -0.09374629706144333, 0.1718449592590332, -0.007803328335285187, 0.0805879533290863, -0.022694628685712814, 0.006300690118223429, 0.033749453723430634, 0.05669882148504257, -0.016320763155817986, 0.015890156850218773, 0.02491586282849312, 0.07135023176670074, 0.04412183165550232, -0.16616013646125793, -0.09480573982000351, -0.05064230039715767, 0.021005000919103622, 0.03640894219279289, 0.016937172040343285, -0.08512598276138306, -0.05171087384223938, -0.03244660422205925, 0.0999913364648819, -0.025397926568984985, -0.0862141102552414, 0.005811893381178379, -0.0826227068901062, 0.008414523676037788, 0.019761299714446068, -0.053309451788663864, 0.022682661190629005, 0.04715321585536003, -0.02739706076681614, -0.07692041248083115, -0.06451194733381271, -0.014175931923091412, 0.06629708409

In [29]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    result = driver.execute_query("""MATCH (n1)-[r:ns1__productFeature]->()
WHERE n1.rdfs__comment_embedding_vector IS NOT NULL
MATCH (n2)-[r2:ns1__productFeature]->()
WHERE n2.rdfs__comment_embedding_vector IS NOT NULL
WITH n1, n2, vector.similarity.euclidean(n2.rdfs__comment_embedding_vector, n1.rdfs__comment_embedding_vector) AS score
RETURN n1, n2, score
ORDER BY score DESCENDING
LIMIT 10""")
    for record in result.records:
        print(record.data())

{'n1': {'rdfs__label_embedding': '{"data": [-0.05175839737057686, 0.027148278430104256, 0.015554022043943405, -0.06239768862724304, -0.07499601691961288, -0.037210676819086075, -0.07269799709320068, -0.04315636307001114, -0.008997028693556786, -0.049477823078632355, 0.05582856759428978, 0.002818681765347719, -0.06939739733934402, -0.03595871850848198, -0.07078132033348083, -0.021209631115198135, 0.05396008864045143, 0.08983342349529266, -0.03903791680932045, 0.06556941568851471, -0.02713705226778984, 0.04042129963636398, -0.04566202312707901, -0.021090330556035042, 0.02332514524459839, -0.0022313850931823254, -0.07620017975568771, 0.05586690083146095, 0.02632870525121689, -0.12457381933927536, 0.0260153915733099, -0.02406284771859646, -0.11851946264505386, -0.039228491485118866, 0.08166897296905518, 0.019759470596909523, -0.09351009875535965, 0.03498615697026253, -0.04858582466840744, 0.018834881484508514, 0.03330003470182419, 0.008101962506771088, -0.024905934929847717, 0.024504574015

In [43]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    result = driver.execute_query("""MATCH (n1)-[r:ns1__productFeature]->()
WHERE n1.rdfs__comment_embedding_vector IS NOT NULL
MATCH (n2)-[r2:ns1__productFeature]->()
WHERE n2.rdfs__comment_embedding_vector IS NOT NULL
WITH n1, n2, vector.similarity.euclidean(n2.rdfs__comment_embedding_vector, n1.rdfs__comment_embedding_vector) AS score
RETURN elementId(n1) AS n1_id, elementId(n2) AS n2_id, score
ORDER BY score DESCENDING
LIMIT 10""")
    for record in result.records:
        print(record.data())

2026-05-11 22:10:04,050 - ERROR - Unable to retrieve routing information
2026-05-11 22:10:04,050 - WARNING - Transaction failed and will be retried in 1.0503558664392958s (Unable to retrieve routing information)
2026-05-11 22:10:05,102 - ERROR - Unable to retrieve routing information
2026-05-11 22:10:05,102 - WARNING - Transaction failed and will be retried in 2.282047110109577s (Unable to retrieve routing information)
2026-05-11 22:10:07,385 - ERROR - Unable to retrieve routing information
2026-05-11 22:10:07,386 - WARNING - Transaction failed and will be retried in 3.9669571338659293s (Unable to retrieve routing information)


KeyboardInterrupt: 

In [30]:
record.data().keys()

dict_keys(['n1', 'n2', 'score'])

In [35]:
result.records

[<Record n1=<Node element_id='4:bef00b61-8293-4758-9d91-4a3ba5aed1c1:481' labels=frozenset({'ns1__Product', 'ns4__ProductType5', 'Resource'}) properties={'rdfs__label_embedding': '{"data": [-0.05175839737057686, 0.027148278430104256, 0.015554022043943405, -0.06239768862724304, -0.07499601691961288, -0.037210676819086075, -0.07269799709320068, -0.04315636307001114, -0.008997028693556786, -0.049477823078632355, 0.05582856759428978, 0.002818681765347719, -0.06939739733934402, -0.03595871850848198, -0.07078132033348083, -0.021209631115198135, 0.05396008864045143, 0.08983342349529266, -0.03903791680932045, 0.06556941568851471, -0.02713705226778984, 0.04042129963636398, -0.04566202312707901, -0.021090330556035042, 0.02332514524459839, -0.0022313850931823254, -0.07620017975568771, 0.05586690083146095, 0.02632870525121689, -0.12457381933927536, 0.0260153915733099, -0.02406284771859646, -0.11851946264505386, -0.039228491485118866, 0.08166897296905518, 0.019759470596909523, -0.09351009875535965,

In [42]:
import pandas as pd


def r_to_df_values(self, qres: any, remove_ns: bool = True):
    first_record = qres.records[0] if qres.records and len(qres.records) > 0 else None
    if not qres.keys and not first_record:
        return pd.DataFrame()
    results = [rec.data() for rec in qres.records]  # type: ignore
    results_df = pd.DataFrame(results)
    # results_df = results_df.map(self.to_readable)
    return results_df


r_to_df_values(None, result)

,n1,n2,score
0,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
1,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
2,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
3,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
4,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
5,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
6,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
7,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
8,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
9,"{'rdfs__label_embedding': '{""data"": [-0.051758...","{'rdfs__label_embedding': '{""data"": [-0.051758...",1.0
